In [57]:
import pathlib
import polars as pl
import chromadb
from chromadb.utils import embedding_functions
from more_itertools import batched
from dotenv import load_dotenv
import openai

In [3]:
def prepare_car_reviews_data(data_path:pathlib.Path,vehicle_years:list[int]):

    dtypes = {
        "": pl.Int64,
        "Review_Date": pl.Utf8,
        "Author_Name": pl.Utf8,
        "Vehicle_Title": pl.Utf8,
        "Review_Title": pl.Utf8,
        "Review": pl.Utf8,
        "Rating": pl.Float64,
    }

    car_reviews = pl.scan_csv(data_path, dtypes=dtypes)

    car_review_db_data = (
        car_reviews.with_columns(
            [
                (
                    pl.col("Vehicle_Title").str.split(
                        by=" ").list.get(0).cast(pl.Int64)
                ).alias("Vehicle_Year"),
                (pl.col("Vehicle_Title").str.split(by=" ").list.get(1)).alias(
                    "Vehicle_Model"
                ),
            ]
        )
        .filter(pl.col("Vehicle_Year").is_in(vehicle_years))
        .select(["Review_Title", "Review", "Rating", "Vehicle_Year", "Vehicle_Model"])
        .sort(["Vehicle_Model", "Rating"])
        .collect()
    )

    ids = [f"review{i}" for i in range(car_review_db_data.shape[0])]
    documents = car_review_db_data["Review"].to_list()
    metadatas = car_review_db_data.drop("Review").to_dicts()

    return {"ids": ids, "documents": documents, "metadatas": metadatas}

In [29]:
def build_chroma_collection(
    chroma_path: pathlib.Path,
    collection_name: str,
    embedding_func_name: str,
    ids: list[str],
    documents: list[str],
    metadatas: list[dict],
    distance_func_name: str = "cosine",
):
    """Create a ChromaDB collection"""

    chroma_client = chromadb.PersistentClient(chroma_path)

    embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name=embedding_func_name
    )

    collection = chroma_client.create_collection(
        name=collection_name,
        embedding_function=embedding_func,
        metadata={"hnsw:space": distance_func_name},
    )

    document_indices = list(range(len(documents)))

    for batch in batched(document_indices, 166):
        start_idx = batch[0]
        end_idx = batch[-1]

        collection.add(
            ids=ids[start_idx:end_idx],
            documents=documents[start_idx:end_idx],
            metadatas=metadatas[start_idx:end_idx],
        )

In [ ]:
DATA_PATH=r'D:\Studying\ai_agents\extra\chromadb-vector-database\data\*'
DATA_PATH=pathlib.Path(DATA_PATH)

CHROMA_PATH = "car_review_embeddings"
EMBEDDING_FUNC_NAME = "multi-qa-MiniLM-L6-cos-v1"
COLLECTION_NAME = "car_reviews"

# chroma_car_reviews_dict = prepare_car_reviews_data(DATA_PATH,2017)

# build_chroma_collection(
# CHROMA_PATH,
# COLLECTION_NAME,
# EMBEDDING_FUNC_NAME,
# chroma_car_reviews_dict["ids"],
# chroma_car_reviews_dict["documents"],
# chroma_car_reviews_dict["metadatas"]
# )

C:\Users\5015023022\AppData\Local\Temp\ipykernel_18492\2056697628.py:13: DeprecationWarning: the argument `dtypes` for `scan_csv` is deprecated. It was renamed to `schema_overrides` in version 0.20.31.
  car_reviews = pl.scan_csv(data_path, dtypes=dtypes)
C:\Users\5015023022\AppData\Local\Temp\ipykernel_18492\2056697628.py:30: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\Studying\ai_agents\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\5015023022\.cache\huggingface\hub\models--sentence-transformers--multi-qa-MiniLM-L6-cos-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [32]:
client = chromadb.PersistentClient(CHROMA_PATH)
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_FUNC_NAME)
collection = client.get_collection(name=COLLECTION_NAME, embedding_function=embedding_func)

In [50]:
load_dotenv(override=True)

True

In [60]:
question = "What's the key to great customer satisfaction?"

great_reviews = collection.query(
query_texts=[question],
n_results=10,
include=["documents"],
where={"Rating": {"$gte": 3}}
)

reviews_str =",".join(great_reviews['documents'][0])

context = f"""
You are a customer success employee at a large
car dealership. Use the following car reviews
to answer questions: {reviews_str}"""

good_review_summaries =  openai.chat.completions.create(
model="gpt-3.5-turbo",
messages=[
{"role": "system", "content": reviews_str},
{"role": "user", "content": question},
],
 temperature=0,
n=1,
)

In [63]:
print(good_review_summaries.choices[0].message.content)

The key to great customer satisfaction lies in delivering on promises, providing exceptional service, and exceeding expectations. It's important to listen to customers, address their needs and concerns promptly, and ensure a positive overall experience. Building trust, being transparent, and showing genuine care for customers can go a long way in fostering loyalty and satisfaction. Consistency in delivering high-quality products and services, along with effective communication, also play a crucial role in achieving great customer satisfaction.
